In [1]:
import json
import data_utils
import conceptset_utils

In [13]:
"""
CLASS_SIM_CUTOFF: Concenpts with cos similarity higher than this to any class will be removed
OTHER_SIM_CUTOFF: Concenpts with cos similarity higher than this to another concept will be removed
MAX_LEN: max number of characters in a concept

PRINT_PROB: what percentage of filtered concepts will be printed
"""

CLASS_SIM_CUTOFF = 0.85
OTHER_SIM_CUTOFF = 0.9
MAX_LEN = 30
PRINT_PROB = 1

dataset = "chestxray"
device = "cuda"

save_name = "data/concept_sets/{}_filtered.txt".format(dataset)

In [14]:
#EDIT these to use the initial concept sets you want

with open("data/concept_sets/gpt3_init/gpt4o_{}_important.json".format(dataset), "r") as f:
    important_dict = json.load(f)
with open("data/concept_sets/gpt3_init/gpt4o_{}_superclass.json".format(dataset), "r") as f:
    superclass_dict = json.load(f)
with open("data/concept_sets/gpt3_init/gpt4o_{}_around.json".format(dataset), "r") as f:
    around_dict = json.load(f)
    
with open(data_utils.LABEL_FILES[dataset], "r") as f:
    classes = f.read().split("\n")

In [15]:
concepts = set()

for values in important_dict.values():
    concepts.update(set(values))

for values in superclass_dict.values():
    concepts.update(set(values))
    
for values in around_dict.values():
    concepts.update(set(values))

print(len(concepts))

466


In [16]:
concepts = conceptset_utils.remove_too_long(concepts, MAX_LEN, PRINT_PROB)

33 Density or attenuation on imaging
37 - Well-defined or ill-defined borders
43 Changes in the skin over the herniated area
40 Loss of lung volume on the affected side
50 Shift of the mediastinum towards the affected side
34 Opacification of the affected area
31 Narrowing of intercostal spaces
31 Displacement of internal organs
31 Increased retrosternal airspace
44 Tracheal deviation (in tension pneumothorax)
32 Fluid in the interlobar fissures
31 Uniform aeration of lung fields
35 Elevation of the left hemidiaphragm
44 Chronic Obstructive Pulmonary Disease (COPD)
36 Loops of bowel in abnormal positions
31 Displacement of lung structures
33 Crowding of pulmonary vasculature
42 Loss of normal sharp contour of the pleura
37 Crowding of ribs on the affected side
89 Silhouette sign (obliteration of normal anatomical borders due to adjacent opacification)
32 Attenuation of pulmonary vessels
40 Decreased lung markings on affected side
55 Impact on adjacent structures (e.g., ribs or diaphrag

In [17]:
concepts = conceptset_utils.filter_too_similar_to_cls(concepts, classes, CLASS_SIM_CUTOFF, device, PRINT_PROB)

366
Class:Consolidation - Deleting Consolidation
Class:Cardiomegaly - Deleting Cardiomegaly
364
Class:Atelectasis - Concept:- Atelectasis, sim:0.923 - Deleting - Atelectasis

Class:Consolidation - Concept:- Consolidation, sim:0.944 - Deleting - Consolidation

Class:Pneumothorax - Concept:- Pneumothorax, sim:0.934 - Deleting - Pneumothorax

Class:Edema - Concept:- Edema, sim:0.935 - Deleting - Edema

Class:Emphysema - Concept:- Emphysema, sim:0.931 - Deleting - Emphysema

Class:Emphysema - Concept:Subcutaneous emphysema, sim:0.859 - Deleting Subcutaneous emphysema

Class:Fibrosis - Concept:- Fibrosis, sim:0.936 - Deleting - Fibrosis

Class:Effusion - Concept:- effusion, sim:0.953 - Deleting - effusion

Class:Pneumonia - Concept:Respiratory infection, sim:0.858 - Deleting Respiratory infection

Class:Pleural_thickening - Concept:- Irregular pleural surface, sim:0.852 - Deleting - Irregular pleural surface

Class:Pleural_thickening - Concept:- Pleural disease, sim:0.885 - Deleting - Pleur

In [18]:
concepts = conceptset_utils.filter_too_similar(concepts, OTHER_SIM_CUTOFF, device, PRINT_PROB)

- Chest wall - Chest wall , sim:0.9476 - Deleting - Chest wall
- Enlarged cardiac silhouette - Enlarged cardiac silhouette , sim:0.9403 - Deleting Enlarged cardiac silhouette
- Hyperinflated lungs - - Hyperinflation of the lungs , sim:0.9239 - Deleting - Hyperinflation of the lungs
- Increased lung opacity - Increased lung opacity , sim:0.9740 - Deleting Increased lung opacity
- Lung - - Lungs , sim:0.9502 - Deleting - Lung
- Lungs - Lungs , sim:0.9136 - Deleting Lungs
- Patchy opacities - Patchy or segmental opacities , sim:0.9031 - Deleting - Patchy opacities
- Pleura - Pleura , sim:0.9157 - Deleting - Pleura
- neoplasm - neoplasm , sim:0.9351 - Deleting - neoplasm
- radiological assessment - - radiological finding , sim:0.9237 - Deleting - radiological assessment
Absence of mediastinal shift - Mediastinal shift , sim:0.9507 - Deleting Absence of mediastinal shift
Absence of pleural effusion - No pleural effusion , sim:0.9609 - Deleting Absence of pleural effusion
Anatomical abnormal

In [19]:
with open(save_name, "w") as f:
    f.write(concepts[0])
    for concept in concepts[1:]:
        f.write("\n" + concept)